# Install and load dependencies

In [ ]:
! pip install -q evaluate
! pip install -qU wandb

In [ ]:
import torch, pprint, evaluate, wandb, os
import numpy as np
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
from huggingface_hub import HfApi, create_repo
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    TrainingArguments,
    AutoModelForSequenceClassification,
    Trainer,
    EarlyStoppingCallback
)
from functools import partial

# Config

In [ ]:
hf_user_name = "amin-oj"
checkpoint = "bert-base-uncased"
checkpoint_name = "sft-bert-base-uncased-mrpc"
push_to_hub = False
num_train_epochs = 10
early_stopping_patience = 5
train_bs = 32
gacc_steps = 2

os.environ["WANDB_LOG_MODEL"] = "end"
os.environ["WANDB_WATCH"] = "false"

fp16=False
# enable mixed precision | faster training and reduced memory usage
# fp16 might introduce computation instability
# use bf16 instead of fp16 if supported by the hardware
bf16 = True

lr_scheduler_type="cosine" # default is linear

# W&B
wandb_entity = "aminojaghi93-amino"
wandb_project = "HFLLM-transformer-finetuning"
wandb_group = "bert-mrpc-analysis"
wandb_run_name = "bf16_P100_cossch_ebs_64"

# HF and W&B Login

In [ ]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api_key)

In [ ]:
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
!hf auth whoami

# Load dataset
- [MRCP Dataset](https://aclanthology.org/)

In [ ]:
raw_datasets = load_dataset("glue", "mrpc")

In [ ]:
print("====== overall split and scehma")
display(raw_datasets)
print("====== a single record")
display(raw_datasets["train"][0])
print("====== what does each label value mean?")
display(raw_datasets["train"].features)
print("====== saved dataset file structure (Apache Arrows File)")
os.listdir("~/.cache/huggingface/datasets/")

# Preprocess the dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

## The memory-inefficient way
- Regardless of the efficiency, we have to pass the sentences as a pair of texts or a pair of lists to the tokenizer.
- This is necessary to get a prediction of whether the two sentences (single or in a batch) are paraphrases or not

### A single pair

In [ ]:
sentences_1 = raw_datasets["train"]["sentence1"]
sentences_2 = raw_datasets["train"]["sentence2"]
inputs = tokenizer(sentences_1[0], sentences_2[0])
print("======== tokenized and encoded input")
pprint.pprint(inputs, compact=True)
print("======== input tokens (by token_type)")
all_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"])
tk_type = inputs["token_type_ids"]
print("type 0 (1st sentence)")
pprint.pprint([tk for tk,type in zip(all_tokens, tk_type) if type==0], compact=True)
print("===")
print("type 1 (2nd sentence)")
pprint.pprint([tk for tk,type in zip(all_tokens, tk_type) if type==1], compact=True)

### List of pairs
text input must be of type
- `str` (single example),
- `list[str]` (batch or single pretokenized example)
- `list[list[str]]` (batch of pretokenized examples).

In [ ]:
tokenized_datasets = tokenizer(
    list(raw_datasets["train"]["sentence1"]),
    list(raw_datasets["train"]["sentence2"]),
    padding=True,
    truncation=True,
)

How can this be optimized?
- We don't have to prep all the data and load it into RAM
    - The dataset is an [Apache Arrow](https://arrow.apache.org/) file stored on disk.
    - It uses memory-mapping to load only the required batch to RAM
    - By using `dataset.map()`, we can keep the result as Apache Arrow.
        -  The `map()` works by applying a function on each element of the dataset
        -  Set `batched = True` to exploit the speed of the backend written in Rust.
            - Not necessary to set `num_proc >=2` if using a fast tokenizer (using the Rust backend) 
    - `with_transform()` is the lazy version of the map
- We don't have to pad all the records in this stage
    - Padding here means we have to pad all records to the largest sequence in the whole dataset (capped by `max_seq_len`)
    - However, we can use *dynamic padding*, which applies padding in every batch inside the `collate` function.
        - This way, all records are padded to the largest element in that specific batch, which saves a lot of GPU VRAM and processing power
        - If you’re training on a TPU it can cause problems — TPUs prefer fixed shapes, even when that requires extra padding.

## The memory-efficient way

In [ ]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    # num_proc = 2, # not necessary if using a rust-backed fast tokenizer
    remove_columns = ['sentence1', 'sentence2', 'idx']
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

`collate` function will
- converts the tokenized_datasets format to `torch.Tensor`
    - Could have been done by passing `return_tensors = "pt` to the tokenizer call
- Concatenate samples into batches
    - (recursively if your elements are lists, tuples, or dictionaries).
- renames `"label"` to `"labels"` (required by the model)
-  It takes a tokenizer when you instantiate it to know
    -  which padding token to use
    -  and whether the model expects padding to be on the left or on the right of the inputs

### Test this new and more efficient prep pipeline

In [ ]:
samples = tokenized_datasets["train"][:8]
samples = {k: v for k, v in samples.items()}
print("========= lengths before collating/padding")
print("input_ids: ", [len(x) for x in samples["input_ids"]])
print("attention_mask: ", [len(x) for x in samples["attention_mask"]])
print("========= lengths after collating/padding")
batch = data_collator(samples)
{k: v.shape for k, v in batch.items()}

# Train

## `Model` and `TrainingArguments` components

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)\

init_training_args = TrainingArguments(
    output_dir = checkpoint_name,
    push_to_hub=push_to_hub,
    num_train_epochs=num_train_epochs,
    report_to = 'none' # disable w&b for now
)

trainer = Trainer(
    model,
    init_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
)

- The `processing_class` parameter tells the Trainer which tokenizer, feature extractor, or image processor to use for processing.
    - Run `??Trainer.__init__` to see more details
- When you pass a tokenizer as the `processing_class`,
    - The default `data_collator` used by the Trainer will be a `DataCollatorWithPadding`.
    - You can skip the data_collator=data_collator line in this case,

## Evaluation component

- We need to build a `compute_metrics()` function and pass it to the `trainer` to evaluate the model
    - It takes an `EvalPrediction` object (a named tuple with a `predictions` field and a `label_ids` field)
    - Returns a dictionary mapping metric names to metric values
        - These are added to the metrics returned by the `trainer.predict()`

In [ ]:
predictions = trainer.predict(tokenized_datasets["validation"])
print("====== trainer prediction result")
print("logits: ", predictions.predictions.shape)
print("labels: ", predictions.label_ids.shape)
print("metrics: ", predictions.metrics.keys())
print("====== evaluation result")
preds = np.argmax(predictions.predictions, axis=-1)
metric = evaluate.load("glue", "mrpc")
metric.compute(predictions=preds, references=predictions.label_ids)

In [ ]:
def compute_metrics(eval_preds, evaluator):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return evaluator.compute(predictions=predictions, references=labels)

## Monitoring component

In [ ]:
wandb.init(
    project=wandb_project,
    group=wandb_group,
    name=wandb_run_name,
)

## Finalized `TrainingArguments`

In [ ]:
steps_per_epoch = tokenized_datasets["train"].num_rows // (train_bs * gacc_steps) + 1
total_steps = steps_per_epoch * num_train_epochs

training_args = TrainingArguments(
    output_dir = checkpoint_name,
    push_to_hub=push_to_hub,
    report_to="wandb",
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=train_bs,
    gradient_accumulation_steps=gacc_steps,
    max_steps=total_steps,
    lr_scheduler_type=lr_scheduler_type,
    eval_strategy="steps", # save and log are "steps" by default
    eval_steps=steps_per_epoch//4,
    save_steps=steps_per_epoch//4,
    logging_steps=10,
    eval_on_start = True,
    logging_first_step=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    fp16 = fp16,
    bf16 = bf16,
    save_safetensors=True,
    save_only_model=True,# if you don’t need optimizer/scheduler state; reduces disk a lot
)

For more advanced training optimization techniques, see the [HF Transformers performance guide](https://huggingface.co/docs/transformers/main/en/performance).

In [ ]:
trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=partial(compute_metrics, evaluator=metric),
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=early_stopping_patience
        )
    ],
)

In [ ]:
trainer.train()
wandb.finish() # necessary in notebooks

# Learning Curve Analysis
- use this [guide](https://huggingface.co/learn/llm-course/chapter3/5#understanding-learning-curves)

# Push to HF Hub

## Download from W&B

In [ ]:
run = wandb.init()
selected_run_name = "model-fp16_P100_cossch_ebs_64:v1"
artifact = run.use_artifact(f'{wandb_entity}/{wandb_project}/{selected_run_name}', type='model')
artifact_dir = artifact.download()
print(f"downloaded directory: {os.listdir(artifact_dir)}")
run.finish()

## Upload to Hub

In [ ]:
HF_REPO_ID = f"{hf_user_name}/{checkpoint_name}"
# ======option 1: creating repo manually ================
# create_repo(repo_id=HF_REPO_ID, exist_ok=True)
# api = HfApi()
# api.upload_folder(
#     repo_id=HF_REPO_ID,
#     folder_path=artifact_dir,
#     repo_type="model",
#     commit_message="Upload checkpoint from W&B run"
# )
# ======option 2: using transformer api ================
tokenizer = AutoTokenizer.from_pretrained(artifact_dir, use_fast=True)  # if tokenizer files exist there

model.push_to_hub(HF_REPO_ID)
model = AutoModelForSequenceClassification.from_pretrained(artifact_dir)
tokenizer.push_to_hub(HF_REPO_ID)

# TODO: creae a more informative model card
    # e.g. 
    # add loss + metrics
    # add architecture and hyperparams
# TODO: search for best practices of managing projects with W&B

# Inference

In [ ]:
from transformers import pipeline
text = """
This was a masterpiece. Not completely faithful to the books,
but enthralling from beginning to end. Might be my favorite of the three.
"""

classifier = pipeline("sentiment-analysis", model=HF_REPO_ID)
classifier(text)